In [2]:
import pandas as pd

In [3]:
raw = pd.read_csv('/content/transactions_raw (2).csv')
display(raw.head())

,transaction_id,transaction_date,merchant_name,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI


In [4]:
merchants = pd.read_csv('/content/merchant_master.csv')
display(merchants.head())

,merchant_id,merchant_name,account_manager,merchant_category,default_region
0,M001,Alpha Mart,Aisha Khan,Grocery,APAC
1,M002,Beta Stores,Rohan Mehta,Electronics,APAC
2,M003,City Pharma,Elena Rossi,Healthcare,EU
3,M004,Delta Travels,Marcus Lee,Travel,US
4,M005,Eco Home,Nina Weber,Home,EU


In [9]:
raw['merchant_name_clean'] = raw['merchant_name'].str.strip().str.lower()
merchants['merchant_name_clean'] = merchants['merchant_name'].str.strip().str.lower()

In [10]:
display(raw.head())
display(merchants.head())

,transaction_id,transaction_date,merchant_name,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method,merchant_name_clean
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,alpha mart
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,alpha mart
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,beta stores
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,beta stores
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,alpha mart


,merchant_id,merchant_name,account_manager,merchant_category,default_region,merchant_name_clean
0,M001,Alpha Mart,Aisha Khan,Grocery,APAC,alpha mart
1,M002,Beta Stores,Rohan Mehta,Electronics,APAC,beta stores
2,M003,City Pharma,Elena Rossi,Healthcare,EU,city pharma
3,M004,Delta Travels,Marcus Lee,Travel,US,delta travels
4,M005,Eco Home,Nina Weber,Home,EU,eco home


In [11]:
master_df = pd.merge(raw, merchants, on='merchant_name_clean', how='left')
master_df = pd.merge(master_df, users, on='user_id', how='left')

In [12]:
display(master_df.head())

,transaction_id,transaction_date,merchant_name_x,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method,merchant_name_clean,merchant_id,merchant_name_y,account_manager,merchant_category,default_region,user_name,signup_date,user_region,risk_tier
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,alpha mart,M001,Alpha Mart,Aisha Khan,Grocery,APAC,Aarav Shah,2026-01-10,India,medium
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,alpha mart,M001,Alpha Mart,Aisha Khan,Grocery,APAC,Meera Iyer,2026-01-18,India,low
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,beta stores,M002,Beta Stores,Rohan Mehta,Electronics,APAC,Kabir Jain,2026-01-22,India,medium
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,beta stores,NaN,NaN,NaN,NaN,NaN,Sofia Rossi,2026-02-01,Italy,low
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,alpha mart,M001,Alpha Mart,Aisha Khan,Grocery,APAC,Aarav Shah,2026-01-10,India,medium


In [13]:
# Convert date columns to datetime objects for proper merging
master_df['transaction_date'] = pd.to_datetime(master_df['transaction_date'])
rates['rate_date'] = pd.to_datetime(rates['rate_date'])

# Merge master_df with rates to get the exchange rate
master_df = pd.merge(master_df, rates, left_on=['transaction_date', 'currency'], right_on=['rate_date', 'currency'], how='left')

# Calculate usd_amount
master_df['usd_amount'] = master_df['raw_amount'] * master_df['usd_rate']

# Drop the redundant 'rate_date' column if desired, or keep for verification
master_df.drop(columns=['rate_date'], inplace=True)

display(master_df.head())

,transaction_id,transaction_date,merchant_name_x,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method,...,merchant_name_y,account_manager,merchant_category,default_region,user_name,signup_date,user_region,risk_tier,usd_rate,usd_amount
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,...,Alpha Mart,Aisha Khan,Grocery,APAC,Aarav Shah,2026-01-10,India,medium,0.0119,4998.0
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,...,Alpha Mart,Aisha Khan,Grocery,APAC,Meera Iyer,2026-01-18,India,low,0.0119,2499.0
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,...,Beta Stores,Rohan Mehta,Electronics,APAC,Kabir Jain,2026-01-22,India,medium,0.0119,6069.0
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,...,NaN,NaN,NaN,NaN,Sofia Rossi,2026-02-01,Italy,low,0.0120,1920.0
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,...,Alpha Mart,Aisha Khan,Grocery,APAC,Aarav Shah,2026-01-10,India,medium,0.0120,4680.0


In [18]:
# Convert 'transaction_date' in gateway and ledger to datetime
gateway['transaction_date'] = pd.to_datetime(gateway['transaction_date'])
ledger['transaction_date'] = pd.to_datetime(ledger['transaction_date'])

# Rename usd_amount to amount_usd in master_df for consistency
# This rename was done in the previous step after calculating usd_amount
# and now master_df contains 'amount_usd'. No need to rename again here.

# Merge master_df with gateway using common merchant_id and transaction_date
master_gateway_df = pd.merge(master_df, gateway, on=['transaction_date', 'merchant_id'], how='left', suffixes=('_raw', '_gateway'))

# Merge master_df with ledger using common merchant_id and transaction_date
master_ledger_df = pd.merge(master_df, ledger, on=['transaction_date', 'merchant_id'], how='left', suffixes=('_raw', '_ledger'))

display(master_gateway_df.head())
display(master_ledger_df.head())

,transaction_id_raw,transaction_date,merchant_name_x,raw_amount,currency,status_raw,risk_score,gateway_region,user_id,payment_method_raw,...,user_name,signup_date,user_region,risk_tier,usd_rate,amount_usd_raw,transaction_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,...,Aarav Shah,2026-01-10,India,medium,0.0119,4998.0,R001,1200.0,success,UPI
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,...,Meera Iyer,2026-01-18,India,low,0.0119,2499.0,R001,1200.0,success,UPI
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,...,Kabir Jain,2026-01-22,India,medium,0.0119,6069.0,R002,900.0,success,Card
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,...,Sofia Rossi,2026-02-01,Italy,low,0.0120,1920.0,NaN,NaN,NaN,NaN
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,...,Aarav Shah,2026-01-10,India,medium,0.0120,4680.0,R003,500.0,success,Wallet


,transaction_id_raw,transaction_date,merchant_name_x,raw_amount,currency,status_raw,risk_score,gateway_region,user_id,payment_method_raw,...,user_name,signup_date,user_region,risk_tier,usd_rate,amount_usd_raw,transaction_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,...,Aarav Shah,2026-01-10,India,medium,0.0119,4998.0,R001,1200.0,success,UPI
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,...,Meera Iyer,2026-01-18,India,low,0.0119,2499.0,R001,1200.0,success,UPI
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,...,Kabir Jain,2026-01-22,India,medium,0.0119,6069.0,R002,850.0,success,Card
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,...,Sofia Rossi,2026-02-01,Italy,low,0.0120,1920.0,NaN,NaN,NaN,NaN
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,...,Aarav Shah,2026-01-10,India,medium,0.0120,4680.0,R003,500.0,success,Wallet


### Reconciliation with Gateway Data

In [15]:
# Identify discrepancies in amount_usd between raw and gateway
amount_discrepancy_gateway = master_gateway_df[
    (master_gateway_df['amount_usd_raw'].notna()) &
    (master_gateway_df['amount_usd_gateway'].notna()) &
    (master_gateway_df['amount_usd_raw'] != master_gateway_df['amount_usd_gateway'])
]
display(amount_discrepancy_gateway[['transaction_id', 'amount_usd_raw', 'amount_usd_gateway']])

# Identify discrepancies in status between raw and gateway
status_discrepancy_gateway = master_gateway_df[
    (master_gateway_df['status_raw'].notna()) &
    (master_gateway_df['status_gateway'].notna()) &
    (master_gateway_df['status_raw'].str.lower() != master_gateway_df['status_gateway'].str.lower())
]
display(status_discrepancy_gateway[['transaction_id', 'status_raw', 'status_gateway']])

,transaction_id,amount_usd_raw,amount_usd_gateway


,transaction_id,status_raw,status_gateway


### Reconciliation with Ledger Data

In [16]:
# Identify discrepancies in amount_usd between raw and ledger
amount_discrepancy_ledger = master_ledger_df[
    (master_ledger_df['amount_usd_raw'].notna()) &
    (master_ledger_df['amount_usd_ledger'].notna()) &
    (master_ledger_df['amount_usd_raw'] != master_ledger_df['amount_usd_ledger'])
]
display(amount_discrepancy_ledger[['transaction_id', 'amount_usd_raw', 'amount_usd_ledger']])

# Identify discrepancies in status between raw and ledger
status_discrepancy_ledger = master_ledger_df[
    (master_ledger_df['status_raw'].notna()) &
    (master_ledger_df['status_ledger'].notna()) &
    (master_ledger_df['status_raw'].str.lower() != master_ledger_df['status_ledger'].str.lower())
]
display(status_discrepancy_ledger[['transaction_id', 'status_raw', 'status_ledger']])

,transaction_id,amount_usd_raw,amount_usd_ledger


,transaction_id,status_raw,status_ledger


### Transactions missing in Gateway or Ledger

In [22]:
# Identify discrepancies in amount_usd between raw and gateway
amount_discrepancy_gateway = master_gateway_df[
    (master_gateway_df['amount_usd_raw'].notna()) &
    (master_gateway_df['amount_usd_gateway'].notna()) &
    (master_gateway_df['amount_usd_raw'] != master_gateway_df['amount_usd_gateway'])
]
display(amount_discrepancy_gateway[['transaction_id_raw', 'amount_usd_raw', 'amount_usd_gateway']])

# Identify discrepancies in status between raw and gateway
status_discrepancy_gateway = master_gateway_df[
    (master_gateway_df['status_raw'].notna()) &
    (master_gateway_df['status_gateway'].notna()) &
    (master_gateway_df['status_raw'].str.lower() != master_gateway_df['status_gateway'].str.lower())
]
display(status_discrepancy_gateway[['transaction_id_raw', 'status_raw', 'status_gateway']])

,transaction_id_raw,amount_usd_raw,amount_usd_gateway
0,T001,4998.0,1200.0
1,T002,2499.0,1200.0
2,T003,6069.0,900.0
4,T005,4680.0,500.0
7,T008,4114.0,950.0
9,T010,7381.0,950.0
12,T013,3720.0,600.0
14,T015,1560.0,600.0
17,T018,1711.0,4100.0


,transaction_id_raw,status_raw,status_gateway
0,T001,captured,success
1,T002,Captured,success
2,T003,CAPTURED,success
4,T005,CAPTURED,success
7,T008,Captured,success
9,T010,captured,success
12,T013,captured,success
14,T015,captured,success
17,T018,chargeback,success


In [23]:
# Identify discrepancies in amount_usd between raw and ledger
amount_discrepancy_ledger = master_ledger_df[
    (master_ledger_df['amount_usd_raw'].notna()) &
    (master_ledger_df['amount_usd_ledger'].notna()) &
    (master_ledger_df['amount_usd_raw'] != master_ledger_df['amount_usd_ledger'])
]
display(amount_discrepancy_ledger[['transaction_id_raw', 'amount_usd_raw', 'amount_usd_ledger']])

# Identify discrepancies in status between raw and ledger
status_discrepancy_ledger = master_ledger_df[
    (master_ledger_df['status_raw'].notna()) &
    (master_ledger_df['status_ledger'].notna()) &
    (master_ledger_df['status_raw'].str.lower() != master_ledger_df['status_ledger'].str.lower())
]
display(status_discrepancy_ledger[['transaction_id_raw', 'status_raw', 'status_ledger']])

,transaction_id_raw,amount_usd_raw,amount_usd_ledger
0,T001,4998.0,1200.0
1,T002,2499.0,1200.0
2,T003,6069.0,850.0
4,T005,4680.0,500.0
7,T008,4114.0,950.0
9,T010,7381.0,950.0
12,T013,3720.0,640.0
14,T015,1560.0,640.0
17,T018,1711.0,4100.0


,transaction_id_raw,status_raw,status_ledger
0,T001,captured,success
1,T002,Captured,success
2,T003,CAPTURED,success
4,T005,CAPTURED,success
7,T008,Captured,success
9,T010,captured,success
12,T013,captured,success
14,T015,captured,success
17,T018,chargeback,success


In [24]:
# Transactions present in raw but missing in gateway
missing_in_gateway = master_gateway_df[master_gateway_df['amount_usd_gateway'].isna()]
display(missing_in_gateway[['transaction_id_raw', 'amount_usd_raw', 'status_raw']])

# Transactions present in raw but missing in ledger
missing_in_ledger = master_ledger_df[master_ledger_df['amount_usd_ledger'].isna()]
display(missing_in_ledger[['transaction_id_raw', 'amount_usd_raw', 'status_raw']])

,transaction_id_raw,amount_usd_raw,status_raw
3,T004,1920.0,failed e05 timeout
5,T006,3300.0,Captured
6,T007,5400.0,chargeback
8,T009,1512.5,captured
10,T011,2359.5,FAILED e05 TIMEOUT
11,T012,3000.0,captured
13,T014,5640.0,captured
15,T016,2596.0,failed E05 timeout
16,T017,2124.0,Failed E05 Timeout
18,T019,3068.0,Failed E05 Timeout


,transaction_id_raw,amount_usd_raw,status_raw
3,T004,1920.0,failed e05 timeout
5,T006,3300.0,Captured
6,T007,5400.0,chargeback
8,T009,1512.5,captured
10,T011,2359.5,FAILED e05 TIMEOUT
11,T012,3000.0,captured
13,T014,5640.0,captured
15,T016,2596.0,failed E05 timeout
16,T017,2124.0,Failed E05 Timeout
18,T019,3068.0,Failed E05 Timeout


In [17]:
# Transactions present in raw but missing in gateway
missing_in_gateway = master_gateway_df[master_gateway_df['amount_usd_gateway'].isna()]
display(missing_in_gateway[['transaction_id', 'amount_usd_raw', 'status_raw']])

# Transactions present in raw but missing in ledger
missing_in_ledger = master_ledger_df[master_ledger_df['amount_usd_ledger'].isna()]
display(missing_in_ledger[['transaction_id', 'amount_usd_raw', 'status_raw']])

,transaction_id,amount_usd_raw,status_raw
0,T001,4998.0,captured
1,T002,2499.0,Captured
2,T003,6069.0,CAPTURED
3,T004,1920.0,failed e05 timeout
4,T005,4680.0,CAPTURED
5,T006,3300.0,Captured
6,T007,5400.0,chargeback
7,T008,4114.0,Captured
8,T009,1512.5,captured
9,T010,7381.0,captured


,transaction_id,amount_usd_raw,status_raw
0,T001,4998.0,captured
1,T002,2499.0,Captured
2,T003,6069.0,CAPTURED
3,T004,1920.0,failed e05 timeout
4,T005,4680.0,CAPTURED
5,T006,3300.0,Captured
6,T007,5400.0,chargeback
7,T008,4114.0,Captured
8,T009,1512.5,captured
9,T010,7381.0,captured


In [5]:
rates = pd.read_csv('/content/exchange_rates.csv')
display(rates.head())

,rate_date,currency,usd_rate
0,2026-03-01,INR,0.0119
1,2026-03-01,EUR,1.0800
2,2026-03-01,USD,1.0000
3,2026-03-02,INR,0.0120
4,2026-03-02,EUR,1.0900


In [6]:
users = pd.read_csv('/content/users.csv')
display(users.head())

,user_id,user_name,signup_date,user_region,risk_tier
0,U001,Aarav Shah,2026-01-10,India,medium
1,U002,Meera Iyer,2026-01-18,India,low
2,U003,Kabir Jain,2026-01-22,India,medium
3,U004,Sofia Rossi,2026-02-01,Italy,low
4,U005,Vikram Nair,2026-02-04,India,medium


In [7]:
gateway = pd.read_csv('/content/gateway.csv')
display(gateway.head())

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [8]:
ledger = pd.read_csv('/content/ledger.csv')
display(ledger.head())

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [28]:
master_df.to_csv('final_fintech_report.csv',index=False)

In [29]:
from google.colab import files
files.download('final_fintech_report.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>